# Day 15 Revision Summary — Week 3 Revision & Architecture Review

- Week 3 end-to-end: Day 11 (pandas data toolkit) → Day 12 (SQL & stratified splits) → Day 13 (linear/logistic regression + metrics) → Day 14 (overfitting, cross-validation, leakage) all combine into one **capstone**: load → split → `Pipeline` → cross-validate → score once on the held-out test set.
- The capstone pattern never changes across models: `StandardScaler` + model inside a `Pipeline`, `cross_val_score(pipe, X_train, y_train, cv=5)` for an honest mean ± std, then `pipe.fit(X_train, y_train)` and a single, final score on `X_test`/`y_test`.
- On `breast_cancer`, the full capstone (scaled Logistic Regression in a Pipeline) reached **CV 0.980 ± 0.013** and test-set **accuracy 0.982, precision 0.986, recall 0.986, F1 0.986** — better than Day 13's unscaled 0.965 accuracy, because scaling (done safely, inside the Pipeline) helped convergence.
- The week's recurring "bug museum": using `and`/`or` instead of `&`/`|` in pandas masks, peeking at the test set during tuning, trusting accuracy alone on imbalanced data, scoring on train instead of test, ignoring the train/test gap, and scaling *before* the split (leakage).
- Architecture through-line: train/test discipline → honest RAG & LLM-as-judge eval sets (Weeks 8 & 10); precision/recall → retrieval-quality metrics; data leakage → benchmark contamination — this week's habits are the eval discipline employers screen for.

*Resource note: the deck's closing homework slide points to `formulas.md` in "today's folder" as the week's formula reference (first-principles derivations) — referenced here, not reproduced, and read as a homework item below.*

## Classwork Exercise 1 — Build the Capstone (`week3_capstone.py`, ~25 min)

**What's being asked:** Type out, from scratch, the full Week 3 capstone: load `breast_cancer` into a DataFrame, stratified 80/20 split, a `StandardScaler` + `LogisticRegression` `Pipeline`, 5-fold cross-validation on the training set, then a single fit + score on the held-out test set. Print the class balance, the CV mean ± std, and the four test metrics. Then (a) remove the `StandardScaler` step and see if the score/convergence changes, and (b) swap the model for a `KNeighborsClassifier` — same pipeline, one line changed. Verified reference numbers: shape `(569, 31)`, classes `{1: 357, 0: 212}`, CV on train `0.980 ± 0.013`, test confusion `[[41, 1], [1, 71]]`, accuracy 0.982, precision 0.986, recall 0.986, F1 0.986.

**Approach:**
1. Load `load_breast_cancer(as_frame=True)`, build `df = data.frame`, and split into `X = df[data.feature_names]`, `y = df["target"]`. Print `df.shape` and the class balance (`df.groupby("target").size()`).
2. Split with `train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)`.
3. Build `Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))])`.
4. Cross-validate on the **training** data only: `cross_val_score(pipe, X_train, y_train, cv=5)`; print mean and std.
5. `pipe.fit(X_train, y_train)`, then `pipe.predict(X_test)` — this is the *one* time you touch the test set.
6. Print `confusion_matrix`, accuracy, precision, recall, F1 on the test predictions.
7. Stretch: remove the `StandardScaler` step and re-run — note any change. Then swap `LogisticRegression` for `KNeighborsClassifier` inside the same pipeline and re-run.

In [ ]:
"""
Day 15 · Classwork Exercise 1 -- The Week 3 capstone
Expected (verified): shape (569, 31), classes {1: 357, 0: 212}
CV on train: ~0.980 +/- 0.013
Test confusion [[41, 1], [1, 71]] ; accuracy 0.982, precision 0.986, recall 0.986, F1 0.986
"""
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# TODO 1: load into a DataFrame, print shape and class balance
data = load_breast_cancer(as_frame=True)
df = data.frame
X, y = None, None  # df[data.feature_names], df["target"]

# TODO 2: stratified, reproducible split

# TODO 3: build the Pipeline: StandardScaler -> LogisticRegression(max_iter=1000)
pipe = None

# TODO 4: cross-validate on X_train, y_train only (cv=5); print mean +/- std
cv_scores = None

# TODO 5: fit the pipeline on the training data, predict on X_test (the ONE look at test)
preds = None

# TODO 6: print confusion matrix, accuracy, precision, recall, F1 on the test predictions

# TODO 7 (stretch): rebuild the pipeline WITHOUT StandardScaler -- does it change much?
# TODO 7b (stretch): swap LogisticRegression for KNeighborsClassifier -- one line changed


## Classwork Exercise 2 — Mixed Drills: Predict, Then Run (~12 min)

**What's being asked:** Before running each line, predict what it will print, then execute and check yourself. Covers the whole week: DataFrame shape, class balance, split sizes, and a rough cross-validation mean — plus a quick judgement call (precision or recall for a cancer screen?).

**Approach:**
1. Load `df = load_breast_cancer(as_frame=True).frame`. Before running, guess `df.shape`; then print it and check.
2. Guess `df.groupby("target").size().to_dict()`; then print and check.
3. Split with `train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)`; guess `Xtr.shape[0]` and `Xte.shape[0]` before printing.
4. Run `cross_val_score(LogisticRegression(max_iter=5000), X, y, cv=5)`; guess the rough mean before printing `round(scores.mean(), 3)`.
5. Answer in a comment: for a cancer screen, should you maximise precision or recall — and why?

In [ ]:
"""
Day 15 · Classwork Exercise 2 -- Mixed drills (predict, then run)
Write your guess as a comment BEFORE each print, then run and compare.
"""
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression

df = load_breast_cancer(as_frame=True).frame

# TODO 1: guess df.shape as a comment, then print(df.shape)

# TODO 2: guess the class balance, then print(df.groupby("target").size().to_dict())

# TODO 3: split (X, y from df) with test_size=0.2, random_state=42, stratify=y;
# guess Xtr.shape[0] and Xte.shape[0] before printing them
X, y = None, None

# TODO 4: run cross_val_score(LogisticRegression(max_iter=5000), X, y, cv=5);
# guess round(scores.mean(), 3) roughly before printing it

# TODO 5: answer as a comment -- cancer screen: maximise precision or recall? why?


## Homework Exercise 1 — Rebuild the Capstone From Memory (~30 min)

**What's being asked:** Rebuild the Week 3 capstone from Classwork Exercise 1 completely from memory, with no notes. Then swap in a second model to prove the pipeline skeleton is reusable.

**Approach:**
1. Without looking at Exercise 1, write the full pipeline again: load → DataFrame → stratified split → `Pipeline(StandardScaler, model)` → cross-validate → fit → single test-set score.
2. Once it runs, compare your numbers against the reference values (CV ≈ 0.980 ± 0.013, test accuracy ≈ 0.982) — investigate any big discrepancy.
3. Swap the classifier for a different one (e.g. `RandomForestClassifier` or `KNeighborsClassifier`) by changing only the `"clf"` step of the pipeline.
4. Re-run cross-validation and the test score with the new model, and print both models' results side by side.

In [ ]:
"""
Day 15 · Homework 1 -- Rebuild the capstone from memory, then swap the model
"""
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# TODO 1: from memory, rebuild the full capstone pipeline (no peeking at Exercise 1!)
# load -> DataFrame -> stratified split -> Pipeline -> cross_val_score -> fit -> test score

# TODO 2: compare your CV mean/std and test accuracy against the reference
# (~0.980 +/- 0.013 CV, ~0.982 test accuracy) -- investigate any big gap

# TODO 3: build a second pipeline with the SAME scaler but a different "clf" step,
# e.g. RandomForestClassifier(n_estimators=200, random_state=42)
pipe_2 = None

# TODO 4: cross-validate and test-score the second pipeline; print both models'
# CV mean +/- std and test accuracy side by side -- which would you ship?


## Homework Exercise 2 — Re-derive Precision & Recall by Hand (~10 min)

**What's being asked:** From the capstone's test confusion matrix (`[[41, 1], [1, 71]]`), re-derive precision and recall by hand, and confirm your numbers match sklearn's `precision_score` / `recall_score` on the same predictions.

**Approach:**
1. Read off `TN, FP, FN, TP = 41, 1, 1, 71` from the capstone's confusion matrix.
2. Compute `precision = TP / (TP + FP)` and `recall = TP / (TP + FN)` by hand.
3. Reuse the capstone's `y_test` and `preds` (from Classwork Exercise 1) to compute sklearn's `precision_score` and `recall_score`.
4. Print both sets of numbers together and confirm they match.

In [ ]:
"""
Day 15 · Homework 2 -- Re-derive precision & recall by hand
Reference confusion matrix: [[41, 1], [1, 71]]
"""
from sklearn.metrics import precision_score, recall_score

# TODO 1: read off TN, FP, FN, TP from the capstone's confusion matrix
TN, FP, FN, TP = 41, 1, 1, 71

# TODO 2: compute precision and recall BY HAND from these four numbers
precision_by_hand = None  # TP / (TP + FP)
recall_by_hand = None     # TP / (TP + FN)

# TODO 3: reuse the capstone's y_test / preds (re-run Exercise 1's pipeline if needed)
# to compute precision_score(y_test, preds) and recall_score(y_test, preds)

# TODO 4: print hand-computed vs sklearn values together -- confirm they match


## Other homework items (no code needed)

- **Read `formulas.md`** in today's folder — the week's formulas (score, generalisation gap, CV mean/std, standardisation, and standard deviation from scratch), explained first-principles.
- **Commit your work:** `git add . && git commit -m "week 3 complete"`.